In [11]:
from typing import Any

from kirin.ir import Method
import matplotlib.pyplot as plt
import numpy as np

from bloqade import squin, tsim
from bloqade.cirq_utils import emit_circuit, load_circuit, noise
from bloqade.pyqrack import StackMemorySimulator
from bloqade.types import MeasurementResult, Qubit
from kirin.dialects.ilist import IList
import matplotlib.pyplot as plt


# this will help us have return types for our methods that have more intuitive names
Register = IList[Qubit, Any]
Measurement = IList[MeasurementResult, Any]

# this function will help us visualize some circuits
def show_circuit(squin_kernel):
    @squin.kernel
    def _to_visualize():
        _ = squin_kernel()

    return tsim.Circuit(_to_visualize).diagram(height=400)


In [12]:
i = 1000

pyqrack_target = StackMemorySimulator(min_qubits=2)

def demo_circuit(squin_kernel):
    circuit = show_circuit(squin_kernel)
    task = pyqrack_target.task(squin_kernel)
    batch_run = task.batch_run(shots=i)
    single_shot = task.run()
    display(circuit, batch_run, single_shot)

## Team Synthesis Part 1: Learn the language of Clifford+ $T$

Learn how to build and simulate simple circuits using **Bloqade Squin** and **Bloqade PyQrack**.

Build a few small 1-qubit and 2-qubit examples. Confirm that you understand how $H, S, T,$ and $CNOT$ act on simple input states.
Use this part to get comfortable creating, simulating, and visualizing circuits constructed from the Clifford+$T := \{H, S, CNOT, T\}$ gateset.

**Goal:** Build intuition for Clifford+T circuits and the simulation workflow.

In [13]:
@squin.kernel
def demo_h() -> Measurement:
	qubits = squin.qalloc(1)
	squin.h(qubits[0])
	bits = squin.broadcast.measure(qubits)
	return bits

demo_circuit(demo_h)

{(<Measurement.Zero: 0>,): 0.516, (<Measurement.One: 1>,): 0.484}

IList([<Measurement.One: 1>])

In [14]:
@squin.kernel
def demo_s() -> Measurement:
	qubits = squin.qalloc(1)
	squin.h(qubits[0])
	squin.s(qubits[0])
	bits = squin.broadcast.measure(qubits)
	return bits

demo_circuit(demo_s)

{(<Measurement.Zero: 0>,): 0.5, (<Measurement.One: 1>,): 0.5}

IList([<Measurement.Zero: 0>])

In [15]:
@squin.kernel
def demo_t() -> Measurement:
	qubits = squin.qalloc(1)
	squin.h(qubits[0])
	squin.t(qubits[0])
	bits = squin.broadcast.measure(qubits)
	return bits

demo_circuit(demo_s)

{(<Measurement.Zero: 0>,): 0.481, (<Measurement.One: 1>,): 0.519}

IList([<Measurement.Zero: 0>])

In [16]:
@squin.kernel
def demo_cx() -> Measurement:
	qubits = squin.qalloc(2)
	squin.h(qubits[0])
	squin.cx(qubits[0], qubits[1])
	bits = squin.broadcast.measure(qubits)
	return bits

demo_circuit(demo_cx)

{(<Measurement.One: 1>, <Measurement.One: 1>): 0.5,
 (<Measurement.Zero: 0>, <Measurement.Zero: 0>): 0.5}

IList([<Measurement.One: 1>, <Measurement.One: 1>])

## Team Synthesis Part 2: Synthesize the rotation family

Focus on the family

$$
R_z(\pi/2^n), \qquad n \in \{0,1,2,3,4,5\}.
$$

How can we implement these “dyadic” $Z$-rotations using only Clifford+$T$?

Try synthesizing these rotations as well as you can using only our chosen gate set for one qubit (yes, only 1 qubit), and different values of $n$. Some implementations may be exact while others may involve approximations, that is okay. It is up to you to explore different synthesis strategies and compare the circuits you find.

We suggest spending time on finding ways to visualize how your approximations act on different initial states, and reflecting on what you tried, how you judged quality, and what changed as the target angle became smaller.

**Goal:** Explore how small $Z$-rotations can be built from Clifford+ $T$ and explain the synthesis strategies you explored.

> **Distance metric**
>
> When comparing a target gate $U$ to an implementation $V$, use the following global-phase-invariant distance:
> 
> $$
> d(U,V)=\sqrt{1-\frac{|\mathrm{Tr}(U^\dagger V)|}{2}}.
> $$
>  
> Interpretation:
> 
> - $d=0$ means exact agreement up to global phase,
> - larger $d$ means a worse approximation.

In [92]:
import numpy as np
from itertools import product


H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
S = np.array([[1, 0], [0, 1j]])
T = np.array([[1, 0], [0, np.exp(1j*np.pi/4)]])

E = T @ H
B = E @ S

GATES = {'H': H, 'S': S, 'T': T, 'E': E, 'B': B}

theta = lambda k: np.pi / (2 ** k)
Rz = lambda theta: np.array([[1, 0], [0, np.exp(1j*theta)]])
distance = lambda U, V: np.sqrt(1 - abs(np.trace(U.conj().T @ V)) / 2)

In [ ]:
def get_sequences(max_N):
	if max_N < 1:
		return None

	old_sequences = [
		(gate, gate == 'T', int(gate == 'S'))
		for gate in GATES.keys()
	]
	new_sequences = []

	while len(old_sequences):
		for (sequence, used_t, s_count) in old_sequences:
			yield sequence

			if len(sequence) == max_N:
				continue

			# B
			if not used_t:
				new_sequences.append((sequence + 'B', False, 1))

			# E
			if not used_t:
				new_sequences.append((sequence + 'E', False, 0))

			# H
			if sequence[-1] not in {'H', 'T', 'E'}:
				new_sequences.append((sequence + 'H', False, 0))

			# T
			if not used_t:
				new_sequences.append((sequence + 'T', True, s_count))

			# S
			if 90 * (s_count + 1) + 45 * used_t < 360 and sequence[-1] != 'E':
				new_sequences.append((sequence + 'S', used_t, s_count + 1))

		old_sequences = new_sequences
		new_sequences = []

	yield 'I'
	return

def find_sequence(theta, max_N):
	U = Rz(theta)

	results = []
	for sequence in get_sequences(max_N):
		V = np.eye(2)
		for gate in sequence:
			if gate != 'I':
				V = V @ GATES[gate]
		results.append((sequence, distance(U, V)))

	return sorted(results, key=lambda item: (item[1], len(item[0])))[:3]

In [80]:
find_sequence(theta(0), 2)

[('SS', np.float64(0.0)),
 ('ST', np.float64(0.27589937928294306)),
 ('TS', np.float64(0.27589937928294306))]

In [81]:
find_sequence(theta(1), 1)

[('S', np.float64(0.0)),
 ('T', np.float64(0.27589937928294306)),
 ('I', np.float64(0.5411961001461969))]

In [82]:
find_sequence(theta(2), 1)

[('T', np.float64(0.0)),
 ('S', np.float64(0.27589937928294306)),
 ('I', np.float64(0.27589937928294306))]

In [99]:
find_sequence(theta(3), 12)

[('EBEEEBEE', np.float64(0.03972140571234099)),
 ('HEBBSEBBSE', np.float64(0.03972140571234099)),
 ('HEBBTSHBBSE', np.float64(0.03972140571234099))]

In [100]:
find_sequence(theta(4), 12)

[('HBEBBBBBSEBB', np.float64(0.01734600620833231)),
 ('HBEBBBBEBBBH', np.float64(0.01734600620833231)),
 ('HBEBBBBEEBHB', np.float64(0.01734600620833231))]

In [101]:
find_sequence(theta(5), 12)

[('SBEEBEEBEEBB', np.float64(0.03263752091438853)),
 ('EBBSBEEBEEBE', np.float64(0.03263752091438853)),
 ('BBSBEEBEEBEE', np.float64(0.03263752091438853))]

## Team Synthesis Part 3: Non-Clifford gates are expensive

How would you approximate the family of rotations above if I now suddenly told you that $T$ gates can no longer be applied to your qubit? If you spent enough time thinking about part 1, the correct answer should be somewhere in the realm of "not good." Unfortunately, this is what often happens in reality (you will learn more about why this happens in the next part). It is up to you to find and implement a protocol to "inject" the effect of $T$ gates onto the main qubit.

To counteract this not-good news, you will now have access to auxiliary qubits on which you will be allowed to apply $T$ gates. You will now also be able to apply $CNOT$ gates using your main qubit as either target or control, as well as across auxiliary qubits. Other than that, it should be just $S$ or $H$ gates on your main qubit. We want you to approximate the rotations on your main qubit and benchmark them using the previous distance metric, working within the bounds of the simulators (i.e. if the circuit cannot be simulated the approximation does not count).

Track the new costs that appear: ancilla count, 2-qubit gate count, circuit depth, repeated trials, feed-forward, or any other relevant overhead.

**Goal:** Rebuild the same 1-qubit rotations in a setting where the non-Clifford gates in the Clifford+ $T$ set must be supplied indirectly.

## Team Synthesis Part 4: Move from one physical qubit to one logical qubit

![steane_code_construction.png](./assets/steane_code_construction.png)

Figure taken from *"Logical quantum processor based on reconfigurable atom arrays"* by *Bluvstein et al.*

Research the $[[7, 1, 3]]$
 Steane code and learn about which gates in the Clifford+ $T$ are "transversal" in this code. Then, construct a kernel that implements the $[[7, 1, 3]]$
 code (note, the circuit above encodes the $|0\rangle$ state) in Squin and use it to represent one logical qubit.

Keeping in mind which gates in the Clifford+$T$ set are transversal and which ones are not, explore how to apply the same family of target rotations on this one logical qubit. This will require careful circuit design, smart synthesis, and some form of injecting $T$-gates (but now onto a logical qubit).

Like in Part 3, benchmark your approximations, track the new costs that appear, and reflect on the overhead that moving from one physical qubit to one logical qubit creates. You can also explore the transition from one physical qubit to one logical qubit in a noisy setting (the tutorial we shared should help you think about how to add noise).

**Goal:** Show how the challenge changes when one physical qubit becomes one logical qubit (optionally with noise).